# Activity 3: From Pixels to Predictions — Classic Image Processing + Machine Learning 🔬
**STIWNU Computing Seminar • November 23, 2026**  
**Colab Runtime:** Default **CPU** &nbsp;•&nbsp; **Time:** ~25 minutes

### Bringing Activities 1 and 2 together
- In **Activity 1**, we predicted from a **table of numbers** (soil, weather).
- In **Activity 2**, a deep learning model **learned by itself** which image patterns matter.

In this activity **you** are the feature engineer. We'll use classic **image processing (OpenCV)** to turn each leaf photo into a row of meaningful numbers (*"how green is it?"*, *"how many brown spots?"*), then train a **Random Forest**, exactly like Activity 1.

```
 Leaf photo ──► Image processing ──► Features (a table!) ──► Random Forest ──► "bean_rust"
              (grayscale, blur,     green_ratio = 0.61
               edges, color masks)  spot_ratio  = 0.18 ...
```

### What you will do
| Step | What happens | Key concept |
|---|---|---|
| 1 | Load images & view the first 2 | Dataset inspection |
| 2 | Grayscale & blur | Noise reduction |
| 3 | Edge detection | Finding outlines & texture |
| 4–5 | Color segmentation in HSV | Detecting disease spots |
| 6–7 | Turn every image into numbers | **Feature extraction** |
| 8–9 | Train & evaluate a Random Forest | Prediction from image features |
| 10 | Predict on new images, step by step | Full pipeline |
| 11 | Compare with deep learning | Choosing the right approach |

> **How to use this notebook:** Run each cell **from top to bottom** with `Shift + Enter`. Read the text cell *before* each code cell; it explains what's about to happen.
>
> 💬 **Think**: a question to discuss with your seatmate &nbsp;•&nbsp; 🧪 **Try it**: change a value and re-run &nbsp;•&nbsp; 💡 **Key idea**: the takeaway to remember

In [ ]:
# ==============================================================================
# SETUP: Import libraries & load the beans dataset (~1–2 minutes)
# ==============================================================================
!pip install -q gradio datasets

import time
import cv2                       # OpenCV, the classic image processing library
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datasets import load_dataset
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, ConfusionMatrixDisplay

beans = load_dataset('AI-Lab-Makerere/beans')   # same dataset as Activity 2 (Hugging Face Hub)
CLASS_NAMES = beans['train'].features['labels'].names
IMG = 256    # work at 256×256 to keep processing fast

def load_split(split):
    images, labels = [], []
    for example in beans[split]:
        img = np.array(example['image'].convert('RGB'))
        images.append(cv2.resize(img, (IMG, IMG), interpolation=cv2.INTER_AREA))
        labels.append(example['labels'])
    return np.array(images), np.array(labels)

train_images, train_labels = load_split('train')
test_images, test_labels = load_split('test')
print(f'✅ Loaded {len(train_images)} training and {len(test_images)} test images')
print('Classes:', CLASS_NAMES)

## Step 1: View the First 2 Images
> ⚠️ **OpenCV gotcha:** OpenCV normally assumes colors are stored as **BGR** (blue first), but our images are **RGB**. That's why every conversion below explicitly says `cv2.COLOR_RGB2...`. Mixing these up is one of the most common image processing bugs!

In [ ]:
# ==============================================================================
# STEP 1: Show the first 2 training images
# ==============================================================================
plt.figure(figsize=(10, 5))
for i in range(2):
    plt.subplot(1, 2, i + 1)
    plt.imshow(train_images[i])
    plt.title(f'Image #{i + 1}: {CLASS_NAMES[train_labels[i]]}')
    plt.axis('off')
plt.tight_layout()
plt.show()

first_two = train_images[:2]

## Step 2: Grayscale & Blur
- **Grayscale** keeps only brightness (1 channel instead of 3). Many algorithms (like edge detection) work on brightness.
- **Gaussian blur** averages each pixel with its neighbors. This removes tiny noise (dust, camera grain) so we detect **real** features.

🧪 **Try it:** Change `BLUR_SIZE` (must be an odd number). What happens with a very large blur?

In [ ]:
# ==============================================================================
# STEP 2: Grayscale + Gaussian blur
# ==============================================================================
BLUR_SIZE = 7   #@param {type:"slider", min:1, max:31, step:2}

fig, axes = plt.subplots(2, 3, figsize=(13, 8.5))
for i, img in enumerate(first_two):
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    blur = cv2.GaussianBlur(gray, (BLUR_SIZE, BLUR_SIZE), 0)
    for ax, im, title in zip(axes[i], [img, gray, blur], ['Original', 'Grayscale', f'Blurred ({BLUR_SIZE}×{BLUR_SIZE})']):
        ax.imshow(im, cmap=None if im.ndim == 3 else 'gray'); ax.set_title(f'#{i + 1} {title}'); ax.axis('off')
plt.tight_layout()
plt.show()

## Step 3: Edge Detection (Canny)
An **edge** is where brightness changes suddenly: leaf outlines, veins, and **the borders of disease spots**. The **Canny** algorithm finds edges using two thresholds:
- Changes **above `HIGH`** are definitely edges.
- Changes **between `LOW` and `HIGH`** count only if connected to a strong edge.

🧪 **Try it:** Lower the thresholds and you get more (noisier) edges; raise them and you get only the strongest outlines.

In [ ]:
# ==============================================================================
# STEP 3: Canny edge detection
# ==============================================================================
LOW = 50    #@param {type:"slider", min:0, max:255, step:5}
HIGH = 150  #@param {type:"slider", min:0, max:255, step:5}

fig, axes = plt.subplots(1, 4, figsize=(18, 4.5))
for i, img in enumerate(first_two):
    gray = cv2.GaussianBlur(cv2.cvtColor(img, cv2.COLOR_RGB2GRAY), (5, 5), 0)
    edges = cv2.Canny(gray, LOW, HIGH)
    axes[2 * i].imshow(img); axes[2 * i].set_title(f'#{i + 1} Original')
    axes[2 * i + 1].imshow(edges, cmap='gray')
    axes[2 * i + 1].set_title(f'#{i + 1} Edges: {(edges > 0).mean():.1%} of pixels')
for ax in axes: ax.axis('off')
plt.tight_layout()
plt.show()

## Step 4: Color Segmentation in HSV, a "Disease Spot Detector" 🎯
RGB is hard to use for "find all brown pixels". The **HSV** color space is easier:
- **H**ue: the *type* of color (OpenCV scale 0–179: red ≈ 0, yellow ≈ 30, green ≈ 60, blue ≈ 120)
- **S**aturation: how vivid the color is
- **V**alue: how bright it is

We create **masks** (black & white images) that mark:
- 🟩 **Green pixels** (hue 35–85), healthy leaf tissue
- 🟫 **Yellow/brown pixels** (hue 5–30), possible disease spots

In [ ]:
# ==============================================================================
# STEP 4: Build green and spot masks using HSV color ranges
# ==============================================================================
GREEN_LOW, GREEN_HIGH = (35, 40, 40), (85, 255, 255)   # (H, S, V) min and max
SPOT_LOW,  SPOT_HIGH  = (5, 50, 40),  (30, 255, 230)

def leaf_masks(img):
    hsv = cv2.cvtColor(img, cv2.COLOR_RGB2HSV)
    green = cv2.inRange(hsv, GREEN_LOW, GREEN_HIGH)     # 255 where pixel is green, else 0
    spots = cv2.inRange(hsv, SPOT_LOW, SPOT_HIGH)       # 255 where pixel is yellow/brown
    return hsv, green, spots

def highlight_spots(img, spots):
    out = img.copy()
    out[spots > 0] = [255, 0, 0]                        # paint suspected spots bright red
    return out

fig, axes = plt.subplots(2, 5, figsize=(20, 8.5))
for i, img in enumerate(first_two):
    hsv, green, spots = leaf_masks(img)
    panels = [(img, None, 'Original'), (hsv[:, :, 0], 'hsv', 'Hue channel'),
              (green, 'gray', 'Green mask'), (spots, 'gray', 'Yellow/brown mask'),
              (highlight_spots(img, spots), None, 'Suspected spots in red')]
    for ax, (im, cmap, title) in zip(axes[i], panels):
        ax.imshow(im, cmap=cmap); ax.set_title(f'#{i + 1} {title}'); ax.axis('off')
plt.tight_layout()
plt.show()

💬 **Think:** Did the spot mask also catch things that **aren't** disease, like soil, dry leaves, or shadows in the background? This is the main weakness of hand-made rules: **the real world is messy**. Keep this in mind for Step 11.

## Step 5: Does the Spot Detector Work Across Classes?
Let's run the detector on 3 random images per class and compute the **spot ratio** = brown pixels ÷ (green + brown pixels).

In [ ]:
# ==============================================================================
# STEP 5: Spot detector on examples from each class
# ==============================================================================
def spot_ratio(img):
    _, green, spots = leaf_masks(img)
    return (spots > 0).sum() / ((green > 0).sum() + (spots > 0).sum() + 1)

rng = np.random.default_rng(0)
fig, axes = plt.subplots(3, 3, figsize=(12, 12.5))
for row, cls in enumerate(range(len(CLASS_NAMES))):
    picks = rng.choice(np.where(train_labels == cls)[0], 3, replace=False)
    for col, idx in enumerate(picks):
        img = train_images[idx]
        axes[row, col].imshow(highlight_spots(img, leaf_masks(img)[2]))
        axes[row, col].set_title(f'{CLASS_NAMES[cls]}\nspot ratio = {spot_ratio(img):.2f}')
        axes[row, col].axis('off')
plt.tight_layout()
plt.show()

## Step 6: Feature Extraction: Turn Every Image into a Row of Numbers 📊
This is the key step. For **every** image we measure:

| Feature | What it measures | Why it might help |
|---|---|---|
| `mean_red/green/blue` | Average color | Diseased leaves are less green |
| `mean_hue/saturation/brightness` | Average HSV values | Color type & vividness |
| `std_hue` | Variety of colors | Spotty leaves have mixed colors |
| `green_ratio` | % of green pixels | Healthy tissue |
| `spot_ratio` | Brown ÷ (green + brown) | Disease spots |
| `edge_density` | % of edge pixels | Spots add many small edges |
| `texture` | Variance of the Laplacian filter | Rough vs. smooth surface |

In [ ]:
# ==============================================================================
# STEP 6: Extract hand-crafted features from every image
# ==============================================================================
def extract_features(img):
    hsv, green, spots = leaf_masks(img)
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    edges = cv2.Canny(cv2.GaussianBlur(gray, (5, 5), 0), 50, 150)
    n_green, n_spots = (green > 0).sum(), (spots > 0).sum()
    return {
        'mean_red':        img[:, :, 0].mean(),
        'mean_green':      img[:, :, 1].mean(),
        'mean_blue':       img[:, :, 2].mean(),
        'mean_hue':        hsv[:, :, 0].mean(),
        'mean_saturation': hsv[:, :, 1].mean(),
        'mean_brightness': hsv[:, :, 2].mean(),
        'std_hue':         hsv[:, :, 0].std(),
        'green_ratio':     n_green / green.size,
        'spot_ratio':      n_spots / (n_green + n_spots + 1),
        'edge_density':    (edges > 0).mean(),
        'texture':         cv2.Laplacian(gray, cv2.CV_64F).var(),
    }

start = time.time()
X_train = pd.DataFrame([extract_features(img) for img in train_images])
X_test  = pd.DataFrame([extract_features(img) for img in test_images])
y_train, y_test = train_labels, test_labels
print(f'✅ Extracted {X_train.shape[1]} features from {len(X_train) + len(X_test)} images in {time.time() - start:.1f} s\n')

print('The first 2 images, now as numbers. This is a table just like Activity 1!')
X_train.head(2).assign(label=[CLASS_NAMES[l] for l in y_train[:2]]).round(3)

## Step 7: Explore the Features
Do the features actually differ between classes? If the boxes below are at different heights, the feature is useful for prediction.

In [ ]:
# ==============================================================================
# STEP 7: Compare feature values across classes
# ==============================================================================
plot_df = X_train.assign(label=[CLASS_NAMES[l] for l in y_train])
fig, axes = plt.subplots(1, 4, figsize=(20, 4.5))
for ax, feat in zip(axes, ['green_ratio', 'spot_ratio', 'edge_density', 'std_hue']):
    sns.boxplot(data=plot_df, x='label', y=feat, ax=ax, hue='label', palette='Greens', legend=False)
    ax.set_title(feat); ax.set_xlabel(''); ax.tick_params(axis='x', rotation=20)
plt.tight_layout()
plt.show()

💬 **Think:** Which feature best separates **healthy** from **diseased** leaves? Is any single feature enough to separate **all 3** classes?

## Step 8: Train a Random Forest & Evaluate
Same algorithm as Activity 1. The model doesn't know these numbers came from images, because to it, they're just a table. We train on the training images and evaluate on the separate **test** images.

In [ ]:
# ==============================================================================
# STEP 8: Train and test a Random Forest on the image features
# ==============================================================================
rf = RandomForestClassifier(n_estimators=300, random_state=42)
rf.fit(X_train, y_train)
y_pred = rf.predict(X_test)
classic_acc = accuracy_score(y_test, y_pred)

print(f'🎯 TEST ACCURACY (classic features + Random Forest): {classic_acc:.2%}\n')
print(classification_report(y_test, y_pred, target_names=CLASS_NAMES))

fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay.from_predictions(y_test, y_pred, display_labels=CLASS_NAMES, cmap='Greens',
                                        ax=ax, colorbar=False, xticks_rotation=20)
ax.set_title('Confusion matrix (test set)')
plt.tight_layout()
plt.show()

## Step 9: Which Image Features Mattered?
Because **we** designed the features, we can explain the model in plain language, e.g. *"it looks mostly at the spot ratio and the color variety."* That's a big advantage of the classic approach.

In [ ]:
# ==============================================================================
# STEP 9: Feature importance
# ==============================================================================
pd.Series(rf.feature_importances_, index=X_train.columns).sort_values().plot(
    kind='barh', color='#2E7D32', figsize=(8, 5))
plt.title('Which hand-crafted image features drive the prediction?')
plt.xlabel('Importance')
plt.tight_layout()
plt.show()

## Step 10: The Full Pipeline on the First 2 Test Images
Photo → image processing → features → prediction, all in one view.

In [ ]:
# ==============================================================================
# STEP 10: Step-by-step prediction for the first 2 test images
# ==============================================================================
probs = rf.predict_proba(X_test)
fig, axes = plt.subplots(2, 4, figsize=(20, 9), gridspec_kw={'width_ratios': [1, 1, 1, 1.1]})
for i in range(2):
    img = test_images[i]
    _, green, spots = leaf_masks(img)
    edges = cv2.Canny(cv2.GaussianBlur(cv2.cvtColor(img, cv2.COLOR_RGB2GRAY), (5, 5), 0), 50, 150)
    f = X_test.iloc[i]
    axes[i, 0].imshow(img); axes[i, 0].set_title(f'1. Test image #{i + 1}\nActual: {CLASS_NAMES[y_test[i]]}')
    axes[i, 1].imshow(highlight_spots(img, spots)); axes[i, 1].set_title(f'2. Spot mask\nspot_ratio = {f.spot_ratio:.2f}')
    axes[i, 2].imshow(edges, cmap='gray'); axes[i, 2].set_title(f'3. Edges\nedge_density = {f.edge_density:.3f}')
    for ax in axes[i, :3]: ax.axis('off')
    colors = ['#2E7D32' if c == y_test[i] else '#BDBDBD' for c in range(len(CLASS_NAMES))]
    axes[i, 3].barh(CLASS_NAMES, probs[i], color=colors); axes[i, 3].set_xlim(0, 1)
    ok = 'CORRECT' if y_pred[i] == y_test[i] else 'WRONG'
    axes[i, 3].set_title(f'4. Prediction ({ok}): {CLASS_NAMES[y_pred[i]]} ({probs[i].max():.0%})')
plt.tight_layout()
plt.show()

## Step 11: Classic Image Processing vs. Deep Learning 🥊
Enter your test accuracy from **Activity 2** below to compare the two approaches side by side.

In [ ]:
# ==============================================================================
# STEP 11: Compare with your Activity 2 (MobileNetV2) result
# ==============================================================================
deep_learning_acc = 0.90   #@param {type:"number"}

comparison = pd.DataFrame({
    'Classic (OpenCV features + Random Forest)': [f'{classic_acc:.0%}', 'You design them by hand', 'Easy: "spot ratio is high"', 'Seconds on CPU', 'Hundreds'],
    'Deep learning (MobileNetV2)':               [f'{deep_learning_acc:.0%}', 'Learned automatically', 'Harder: a "black box"', 'Minutes on GPU', 'Hundreds (with transfer learning)'],
}, index=['Test accuracy', 'Features', 'Explainability', 'Training time', 'Images needed'])
comparison

💡 **Key idea:** Deep learning usually wins on accuracy for messy real-world photos, because it learns features that are hard to describe with rules. Classic image processing is **faster, explainable, and runs on tiny devices**, and it's still widely used for **pre-processing**, **measurement** (e.g., *"what % of the leaf is infected?"*), and controlled settings like factory inspection. **The best capstone projects often combine both.**

## 🧪 Challenge: Beat the Baseline!
Work in pairs. Add **one new feature** to `extract_features_v2` below and see if test accuracy goes up. Ideas:
- `yellow_ratio`: pixels with hue 20–30 only (yellowing)
- `dark_ratio`: very dark pixels (`hsv[:, :, 2] < 60`), for dead tissue
- `num_spots`: number of separate spot blobs, using `cv2.connectedComponents(spots)`
- Tweak the HSV ranges in Step 4 and re-run from there

In [ ]:
# ==============================================================================
# CHALLENGE: Add your own feature(s) and compare accuracy
# ==============================================================================
def extract_features_v2(img):
    feats = extract_features(img)
    hsv, green, spots = leaf_masks(img)
    # 👇 Example new feature: number of separate spot blobs. Add your own below!
    n_blobs, _ = cv2.connectedComponents(cv2.morphologyEx(spots, cv2.MORPH_OPEN, np.ones((3, 3), np.uint8)))
    feats['num_spots'] = n_blobs - 1
    # feats['dark_ratio'] = (hsv[:, :, 2] < 60).mean()
    return feats

X_train2 = pd.DataFrame([extract_features_v2(img) for img in train_images])
X_test2  = pd.DataFrame([extract_features_v2(img) for img in test_images])
rf2 = RandomForestClassifier(n_estimators=300, random_state=42).fit(X_train2, y_train)
new_acc = accuracy_score(y_test, rf2.predict(X_test2))
print(f'Baseline accuracy : {classic_acc:.2%}')
print(f'Your new accuracy : {new_acc:.2%}  {"🎉 Improved!" if new_acc > classic_acc else "Keep trying!"}')

## Bonus: Try Your Own Leaf Photo 📸
Upload any leaf photo (or take one with your phone) and watch the full pipeline: masks, edges, and the prediction.

> If the venue network blocks the public link, change `share=True` to `share=False`.

In [ ]:
# ==============================================================================
# BONUS: Interactive pipeline viewer (Gradio)
# ==============================================================================
import gradio as gr

def analyze(image):
    if image is None:
        return None, None, {}
    img = cv2.resize(np.array(image.convert('RGB')), (IMG, IMG), interpolation=cv2.INTER_AREA)
    _, green, spots = leaf_masks(img)
    edges = cv2.Canny(cv2.GaussianBlur(cv2.cvtColor(img, cv2.COLOR_RGB2GRAY), (5, 5), 0), 50, 150)
    p = rf.predict_proba(pd.DataFrame([extract_features(img)]))[0]
    return highlight_spots(img, spots), edges, {CLASS_NAMES[i]: float(p[i]) for i in range(len(CLASS_NAMES))}

gr.Interface(
    fn=analyze,
    inputs=gr.Image(type='pil', label='Leaf photo'),
    outputs=[gr.Image(label='Suspected spots (red)'), gr.Image(label='Edges'), gr.Label(label='Prediction')],
    title='🔬 From Pixels to Predictions: Classic Leaf Analyzer',
).launch(share=True)

## ✅ Recap
1. **Image processing** (grayscale, blur, edges, color masks) turns pixels into **meaningful measurements**.
2. **Feature extraction** converts images into a **table**, and then any prediction model from Activity 1 works.
3. **HSV** makes color-based segmentation much easier than RGB.
4. Hand-crafted features are **explainable and fast** but can be fooled by messy backgrounds; deep learning **learns** features but is harder to explain.

### 💬 Check your understanding
1. Why did we convert to HSV instead of using RGB directly to find brown spots?
2. Give one reason the spot detector might flag a **healthy** leaf as diseased.
3. For your own project idea, which approach would you choose (classic, deep learning, or both), and why?